# Running this experiment

See README.md for setup, required inputs, and limitations. Run cells in order from a fresh kernel. Saved outputs have been cleared; historical metrics are not newly verified results. Environment installation and external training commands are manual setup steps.

In [ ]:
from project_utils import project_path

# Override these paths for your local datasets. PROJECT_DATA_DIR changes their base.
INPUT_PATHS = {
    "CVD_CSV": project_path("CVD.csv"),
    "X_TRAIN_SCALED_BALANCED_CSV": project_path("X_train_scaled_balanced.csv"),
    "Y_TRAIN_BALANCED_CSV": project_path("y_train_balanced.csv"),
    "X_TEST_SCALED_CSV": project_path("X_test_scaled.csv"),
    "Y_TEST_CSV": project_path("y_test.csv"),
}

# Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.over_sampling import SMOTE

from sklearn.preprocessing import LabelEncoder

# Load Dataset

In [ ]:
df = pd.read_csv(INPUT_PATHS["CVD_CSV"])
df.head(10)

Our Target Variable will be the Diabetes column

# Data Preprocessing

## Get Stats

In [ ]:
df.describe()
# describe only shows statistics for the numerical variables and not for the categorical variables

In [ ]:
df.shape

308,854 rows and 19 columns

## Check for missing values

In [ ]:
missing_values = df.isnull().sum()
print(missing_values)

No missing values

## Check Categorical Values

In [ ]:
categorical_columns = df.select_dtypes(include=["object", "string", "category"]).columns
print(len(categorical_columns))
for col in categorical_columns:
    print(f"Distribution of '{col}':")
    print(df[col].value_counts())
    print("\n")

## Check for Duplicate Values

In [ ]:
# check duplicate values
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.shape

80 Duplicate Values removed

## Merge 4 classes into 2

In [ ]:
df["Diabetes"] = df["Diabetes"].replace(
    {
        "No, pre-diabetes or borderline diabetes": "No",
        "Yes, but female told only during pregnancy": "Yes",
    }
)

# Check if the outcome variable is now binary
print(df["Diabetes"].value_counts())

## Encode Categorical Values

In [ ]:
# Create a copy of the DataFrame to avoid modifying the original
df_encoded = df.copy()

# Create a label encoder object
label_encoder = LabelEncoder()

# Iterate through each object column and encode its values
for column in df_encoded.select_dtypes(include=["object", "string"]):
    df_encoded[column] = label_encoder.fit_transform(df_encoded[column])

# Now, df_encoded contains the label-encoded categorical columns
df_encoded.head()

## Check Correlation Matrix

In [ ]:
# Calculate the correlation matrix for Data
correlation_matrix = df_encoded.corr(numeric_only=True)

# Create a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

There's a 70% correlation between Height and Sex, and 86% correlation between BMI and Weight

## Feature Selection: Drop Height and Weight as they have high correlation to Sex and BMI

In [ ]:
# Assuming df is your DataFrame
df_encoded = df_encoded.drop(["Height_(cm)", "Weight_(kg)"], axis=1)

print(df_encoded.columns)

## Deal with Outliers

In [ ]:
# Define the columns to remove outliers (numerical value columns)
selected_columns = [
    "BMI",
    "Alcohol_Consumption",
    "Fruit_Consumption",
    "Green_Vegetables_Consumption",
    "FriedPotato_Consumption",
]

# Calculate the IQR for the selected columns in the training data
Q1 = df_encoded[selected_columns].quantile(0.25)
Q3 = df_encoded[selected_columns].quantile(0.75)
IQR = Q3 - Q1

# SetTING a threshold value for outlier detection (e.g., 1.5 times the IQR)
threshold = 3

# CreatING a mask for outliers in the selected columns
outlier_mask = (
    (df_encoded[selected_columns] < (Q1 - threshold * IQR))
    | (df_encoded[selected_columns] > (Q3 + threshold * IQR))
).any(axis=1)

# Remove rows with outliers from X_train and y_train
df_encoded_cleaned = df_encoded[~outlier_mask]

# Print the number of rows removed
num_rows_removed = len(df_encoded) - len(df_encoded_cleaned)
print(f"Number of rows removed due to outliers: {num_rows_removed}")

## Visualization

In [ ]:
target_column = "Diabetes"  # Replace 'Diabetes' with the actual column name if different
feature_columns = [col for col in df.columns if col != target_column]

# Set a Seaborn style for better aesthetics
sns.set(style="whitegrid")

# Generate box plots with enhanced visualizations
for feature in feature_columns:
    plt.figure(figsize=(12, 7))  # Larger plot for better readability
    sns.boxplot(
        x=target_column,
        y=feature,
        data=df,
        palette="Set3",  # Use a color palette for better contrast
        showfliers=False,  # Optionally hide outliers for a clearer plot
    )
    sns.stripplot(
        x=target_column,
        y=feature,
        data=df,
        color="black",
        alpha=0.3,
        jitter=True,
        size=4,
    )
    plt.title(f"{feature} vs {target_column}", fontsize=16)
    plt.xlabel(target_column, fontsize=14)
    plt.ylabel(feature, fontsize=14)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
selected_variables = "Diabetes"  # Replace 'Diabetes' with the actual column name if different
disease_conditions = [col for col in df_encoded.columns if col != target_column]

for disease in disease_conditions:
    plt.figure(figsize=(10, 4))
    sns.countplot(data=df, x="Diabetes", hue=disease)
    plt.title("Relationship between " + "Diabetes" + " and " + disease)
    plt.xticks(rotation=90)
    plt.show()

## Divide Dataset to Train and Test Sets and Scale Them


In [ ]:
X = df_encoded.drop("Diabetes", axis=1)
y = df_encoded["Diabetes"]

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Example dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler on the training data
scaler.fit(X_train)

# Transform both train and test sets
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

## Deal with Class Imbalance with SMOTE


In [ ]:
# CHECK THE CLASS VARIABLE
y_train.value_counts()

In [ ]:
smote = SMOTE(random_state=42)
X_train_scaled_balanced, y_train_balanced = smote.fit_resample(X_train_scaled_df, y_train)

In [ ]:
y_train_balanced.value_counts()

In [ ]:
y_test.value_counts()

In [ ]:
X_train_scaled_balanced.head()

In [ ]:
X_test_scaled_df

All classes are balanced

## Save the files

In [ ]:
pd.DataFrame(X_train_scaled_balanced).to_csv(
    INPUT_PATHS["X_TRAIN_SCALED_BALANCED_CSV"], index=False
)
pd.DataFrame(y_train_balanced).to_csv(INPUT_PATHS["Y_TRAIN_BALANCED_CSV"], index=False)

# Save the test data
pd.DataFrame(X_test_scaled_df).to_csv(INPUT_PATHS["X_TEST_SCALED_CSV"], index=False)
pd.DataFrame(y_test).to_csv(INPUT_PATHS["Y_TEST_CSV"], index=False)

print("All files have been saved as CSVs.")